# Saudi Hip Fracture Demand to 2030 — Notebook 1: Data preparation

**Question:** How many hip fractures will Saudi citizens have in 2030, and what does that mean for implants (nails vs hip replacements) and rehabilitation?

**How accuracy is proven:** every step has gates – automatic checks that stop the notebook if a result does not match its source. At the end of this notebook, a final notebook gate re-reads the raw files and the saved clean files and checks them against each other.

## Step 1 – Load every raw file and check it against its own source totals
Eight raw files are used. Before any cleaning, each one is loaded and checked against totals published in the source itself, so we know nothing was lost or mistyped when the files were downloaded or extracted from PDFs.

| File | Source | What is checked |
|---|---|---|
| census2022_age_sex_nationality.csv | GASTAT census 2022 (RCRC open data) | Saudi total = 18,792,262 · all residents = 32,175,224 · 111 single ages for each sex and nationality |
| un_wpp_saudi_2015_2030_age_sex.csv | UN World Population Prospects 2024 | 16 years × 2 sexes × 21 age groups = 672 rows · 2022 total ≈ census total |
| glmm_saudi_pop_agegroup_2010_2022.csv | GASTAT via GLMM (PDF) | age groups add up to the TOTAL row in every year · 2022 = census |
| hipfracture_rates_table1.csv | Al-Saleh et al. 2022, Table 1 | counts add up to 296 men and 388 women · rates recomputed from counts match the printed rates |
| hipfracture_published_projections_table2.csv | Same paper, Table 2 | men + women = total in every year |
| hipfracture_sites.csv | Same paper, Results | fracture sites add up to 100% |

In [8]:
# Starting: I connect Google Drive and load the libraries I need
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# Now I define the project folders: raw data stays untouched, clean data and figures are saved separately
BASE   = '/content/drive/MyDrive/hip-fracture-saudi'
RAW    = f'{BASE}/data/raw'
MANUAL = f'{RAW}/manual_tables'
CLEAN  = f'{BASE}/data/clean'
FIG    = f'{BASE}/figures'
for folder in [CLEAN, FIG]:
    os.makedirs(folder, exist_ok=True)

# Now I write my gate: it stops the notebook on failure and keeps a log of every passed check
GATE_LOG = []
def gate(condition, message):
    if not condition:
        raise AssertionError('GATE FAILED: ' + message)
    GATE_LOG.append(message)
    print('GATE PASSED:', message)

# Now I check that all eight raw files are where I saved them
FILES = {
    'census': f'{RAW}/census2022_age_sex_nationality.csv',
    'un':     f'{RAW}/un_wpp_saudi_2015_2030_age_sex.csv',
    'paper':  f'{RAW}/hipfracture_saudi_2022_paper.pdf',
    'glmm_pdf': f'{RAW}/glmm_saudi_pop_agegroup_2010_2022.pdf',
    'glmm':   f'{MANUAL}/glmm_saudi_pop_agegroup_2010_2022.csv',
    'rates':  f'{MANUAL}/hipfracture_rates_table1.csv',
    'table2': f'{MANUAL}/hipfracture_published_projections_table2.csv',
    'sites':  f'{MANUAL}/hipfracture_sites.csv',
}
missing = [name for name, path in FILES.items() if not os.path.exists(path)]
gate(not missing, f'all 8 raw files found (missing: {missing or "none"})')

# ---------- 1.1 Census 2022 ----------
# Now I load the census and keep only the whole-Kingdom rows ("Total (KSA Govs.)") to check its totals
census = pd.read_csv(FILES['census'], encoding='utf-8-sig')
ksa = census[census['Governorate Code'] == 'TOT_KSA']
saudi_total = ksa.loc[ksa['Citizenship'] == 'Saudi', 'Observation value'].sum()
all_total   = ksa['Observation value'].sum()
gate(saudi_total == 18_792_262, f'census: Saudi citizens in 2022 = {saudi_total:,} (GASTAT 18,792,262)')
gate(all_total == 32_175_224,   f'census: all residents in 2022 = {all_total:,} (GASTAT 32,175,224)')
ages_per_group = ksa.groupby(['Citizenship', 'Gender'])['Age'].nunique()
gate((ages_per_group == 111).all() and len(ages_per_group) == 4,
     'census: 111 single ages (0 to 110) for each sex and nationality')

# ---------- 1.2 UN population ----------
# Now I load the UN file and check it is complete: 16 years x 2 sexes x 21 age groups
un = pd.read_csv(FILES['un'])
gate(len(un) == 16 * 2 * 21, f'UN: {len(un)} rows = 16 years x 2 sexes x 21 age groups')
gate(sorted(un['Time'].unique()) == list(range(2015, 2031)), 'UN: every year 2015-2030 present')
gate(set(un['Sex']) == {'Male', 'Female'}, 'UN: male and female only')
gate(not un.duplicated(['Time', 'Sex', 'Age']).any(), 'UN: no duplicate year/sex/age rows')
un_2022 = un.loc[un['Time'] == 2022, 'Value'].sum()
gap = abs(un_2022 - all_total) / all_total * 100
gate(gap < 0.01, f'UN: 2022 total {un_2022:,.0f} agrees with the census {all_total:,} (difference {gap:.4f}%)')

# ---------- 1.3 GASTAT age groups 2010-2022 (typed from the GLMM PDF) ----------
# Now I check that in every year the age groups add up to the TOTAL row printed in the PDF
glmm = pd.read_csv(FILES['glmm'])
years = [c for c in glmm.columns if c.startswith('y')]
body, total_row = glmm[glmm['age_group'] != 'TOTAL'], glmm[glmm['age_group'] == 'TOTAL']
gate((body[years].sum() == total_row[years].iloc[0]).all(), f'GLMM: age groups add up to the TOTAL row in all {len(years)} years')
gate(total_row['y2022'].iloc[0] == saudi_total, 'GLMM: 2022 total equals the census Saudi total')

# ---------- 1.4 Hip fracture rates (paper, Table 1) ----------
# Now I check the typed counts against the paper's totals and recompute every rate from the counts
rates = pd.read_csv(FILES['rates'])
rbody, rtot = rates[rates['age_group'] != 'TOTAL'], rates[rates['age_group'] == 'TOTAL'].iloc[0]
for col in ['population_2017_male', 'population_2017_female', 'fractures_2017_2018_male', 'fractures_2017_2018_female']:
    gate(rbody[col].sum() == rtot[col], f'paper Table 1: {col} adds up to the printed total ({int(rtot[col]):,})')
gate(rtot['fractures_2017_2018_male'] + rtot['fractures_2017_2018_female'] == 684, 'paper: 296 men + 388 women = 684 fractures')
for sex in ['male', 'female']:
    # the study counted 2 years of fractures, so annual rate = fractures / 2 / population x 100,000
    recomputed = rbody[f'fractures_2017_2018_{sex}'] / 2 / rbody[f'population_2017_{sex}'] * 100_000
    printed = rbody[f'incidence_per_100000_{sex}']
    ok = (recomputed - printed).abs() <= (0.03 * printed).clip(lower=0.5)   # printed rates are rounded
    gate(ok.all(), f'paper Table 1: every {sex} rate recomputed from the counts matches the printed rate (within rounding)')

# ---------- 1.5 Published projections (paper, Table 2) and fracture sites ----------
# Finally I check the paper's projection table and the fracture-site split
table2 = pd.read_csv(FILES['table2']).set_index('sex')
yrs2 = [c for c in table2.columns if c.startswith('y')]
gate((table2.loc['Men', yrs2] + table2.loc['Women', yrs2] == table2.loc['Men and women', yrs2]).all(),
     'paper Table 2: men + women = total in every projected year')
gate(table2.loc['Men and women', 'y2015'] == 2949, 'paper Table 2: published 2015 estimate is 2,949')
sites = pd.read_csv(FILES['sites'])
gate(sites['percent'].sum() == 100, 'paper: fracture sites add up to 100%')

print(f'\nStep 1 complete: {len(GATE_LOG)} gates passed, 0 failed')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GATE PASSED: all 8 raw files found (missing: none)
GATE PASSED: census: Saudi citizens in 2022 = 18,792,262 (GASTAT 18,792,262)
GATE PASSED: census: all residents in 2022 = 32,175,224 (GASTAT 32,175,224)
GATE PASSED: census: 111 single ages (0 to 110) for each sex and nationality
GATE PASSED: UN: 672 rows = 16 years x 2 sexes x 21 age groups
GATE PASSED: UN: every year 2015-2030 present
GATE PASSED: UN: male and female only
GATE PASSED: UN: no duplicate year/sex/age rows
GATE PASSED: UN: 2022 total 32,175,352 agrees with the census 32,175,224 (difference 0.0004%)
GATE PASSED: GLMM: age groups add up to the TOTAL row in all 13 years
GATE PASSED: GLMM: 2022 total equals the census Saudi total
GATE PASSED: paper Table 1: population_2017_male adds up to the printed total (260,453)
GATE PASSED: paper Table 1: population_2017_female adds up to the printed total (25

## Step 2 – Clean the 2022 census into one population table by age, sex and nationality
The census file contains rows for Riyadh, its governorates, the rest of the country and the whole Kingdom. I keep only the **whole-Kingdom rows** ("Total (KSA Govs.)") and turn the age codes (`Y000`…`Y110`) into numbers. The result is one tidy table: **age (0–110) × sex × nationality → population**. Saudi citizens are the main population of this study; non-Saudis are kept in the same table for a separate, clearly labelled line later.

**Gates:**
- 111 ages × 2 sexes × 2 nationalities = 444 rows, no duplicates
- every total is unchanged by cleaning (Saudi, non-Saudi, male, female)
- my own 5-year groups equal the census's 5-year column
- Saudi age groups 45–64 and 65+ equal the separate GASTAT/GLMM 2022 table (an independent source)
- the saved file, read back from Drive, is identical

In [9]:
# Starting: I keep only the whole-Kingdom rows of the census (not Riyadh or the other sub-totals)
ksa = census[census['Governorate Code'] == 'TOT_KSA'].copy()

# Now I turn the age code into a number: 'Y000' -> 0, 'Y085' -> 85, 'Y110' -> 110
ksa['age'] = ksa['Age'].str.replace('Y', '', regex=False).astype(int)

# Now I keep only the columns I need, with simple names
pop = (ksa.rename(columns={'Gender': 'sex', 'Citizenship': 'nationality', 'Observation value': 'population'})
          [['nationality', 'sex', 'age', 'population']]
          .sort_values(['nationality', 'sex', 'age'])
          .reset_index(drop=True))
pop['sex'] = pop['sex'].str.lower()                                   # 'male' / 'female'
pop['nationality'] = pop['nationality'].replace({'Non-Saudi': 'non-saudi', 'Saudi': 'saudi'})

# Now I check the shape: every age, sex and nationality appears exactly once
gate(len(pop) == 111 * 2 * 2, f'clean table has {len(pop)} rows = 111 ages x 2 sexes x 2 nationalities')
gate(not pop.duplicated(['nationality', 'sex', 'age']).any(), 'no duplicate age/sex/nationality rows')
gate(pop['age'].min() == 0 and pop['age'].max() == 110, 'ages run from 0 to 110')

# Now I prove cleaning lost nothing: every total equals the raw census
raw_tot = ksa.groupby(['Citizenship', 'Gender'])['Observation value'].sum()
for (nat, sex), raw_value in raw_tot.items():
    clean_value = pop.loc[(pop['nationality'] == nat.lower()) & (pop['sex'] == sex.lower()), 'population'].sum()
    gate(clean_value == raw_value, f'{nat} {sex}: {clean_value:,} = raw census {raw_value:,}')
gate(pop['population'].sum() == 32_175_224, 'grand total unchanged: 32,175,224')

# Now I build my own 5-year groups and compare them with the census's own 5-year column
pop['age_start_5y'] = (pop['age'] // 5 * 5).clip(upper=100)          # 100+ is the last group
mine = pop.groupby(['nationality', 'sex', 'age_start_5y'])['population'].sum()
ksa['age_start_5y'] = ksa['Age in 5-year groups'].str[1:4].astype(int)
theirs = ksa.groupby([ksa['Citizenship'].str.lower(), ksa['Gender'].str.lower(), 'age_start_5y'])['Observation value'].sum()
theirs.index.names = mine.index.names
gate(mine.equals(theirs.sort_index()), 'my 5-year groups equal the census 5-year column in every cell')

# Now I compare Saudi older age groups with the independent GASTAT/GLMM 2022 table
saudi5 = pop[pop['nationality'] == 'saudi'].groupby('age_start_5y')['population'].sum()
glmm22 = glmm.set_index('age_group')['y2022']
for start in [45, 50, 55, 60]:
    label = f'Y{start:03d}-{start + 4:03d}'
    gate(saudi5[start] == glmm22[label], f'Saudi {label}: census {saudi5[start]:,} = GASTAT/GLMM {glmm22[label]:,}')
saudi65 = saudi5[saudi5.index >= 65].sum()
gate(saudi65 == glmm22['Y065+'], f'Saudi 65+: census {saudi65:,} = GASTAT/GLMM {glmm22["Y065+"]:,}')

# Now I save the clean table and read it back to prove the file on Drive is complete
out = f'{CLEAN}/pop2022_by_age_sex_nationality.csv'
pop.to_csv(out, index=False)
check = pd.read_csv(out)
gate(check.equals(pop), f'saved file read back from Drive is identical ({check.shape[0]} rows x {check.shape[1]} columns)')

# Finally I show the Saudi population aged 45+ by 5-year group (the ages that matter for hip fractures)
view = (pop[(pop['nationality'] == 'saudi') & (pop['age'] >= 45)]
          .pivot_table(index='age_start_5y', columns='sex', values='population', aggfunc='sum'))
view['total'] = view.sum(axis=1)
view.index = [f'{a}-{a + 4}' if a < 100 else '100+' for a in view.index]
print('\nSaudi citizens aged 45+ in 2022:')
print(view.to_string())
print(f'\nSaudi citizens aged 50+: {view.loc[view.index[1]:, "total"].sum():,}')
print(f'\nStep 2 complete: {len(GATE_LOG)} gates passed so far, 0 failed')

GATE PASSED: clean table has 444 rows = 111 ages x 2 sexes x 2 nationalities
GATE PASSED: no duplicate age/sex/nationality rows
GATE PASSED: ages run from 0 to 110
GATE PASSED: Non-Saudi Female: 3,138,498 = raw census 3,138,498
GATE PASSED: Non-Saudi Male: 10,244,464 = raw census 10,244,464
GATE PASSED: Saudi Female: 9,358,131 = raw census 9,358,131
GATE PASSED: Saudi Male: 9,434,131 = raw census 9,434,131
GATE PASSED: grand total unchanged: 32,175,224
GATE PASSED: my 5-year groups equal the census 5-year column in every cell
GATE PASSED: Saudi Y045-049: census 785,913 = GASTAT/GLMM 785,913
GATE PASSED: Saudi Y050-054: census 653,994 = GASTAT/GLMM 653,994
GATE PASSED: Saudi Y055-059: census 529,004 = GASTAT/GLMM 529,004
GATE PASSED: Saudi Y060-064: census 390,642 = GASTAT/GLMM 390,642
GATE PASSED: Saudi 65+: census 659,871 = GASTAT/GLMM 659,871
GATE PASSED: saved file read back from Drive is identical (444 rows x 5 columns)

Saudi citizens aged 45+ in 2022:
sex    female    male   tota

## Step 3 – Clean the hip fracture rates and decide how to treat the oldest age group
The paper gives annual hip fracture rates per 100,000 for men and women in 5-year age groups from 45. Its last row is printed as **"80–84"**, but the evidence says it is really **80 and over**:
- in the paper, that row's population is *larger* than the 75–79 row – impossible for a 5-year group in a population where each older group is smaller;
- the table stops there, although the text says fracture rates keep rising up to age 90.

I test this with the census instead of assuming it: the ratio between the paper's last row and its 75–79 row is compared with the census ratio for **80–84 ÷ 75–79** and for **80+ ÷ 75–79**. The closer match decides how the row is used.

Then every single age from 45 to 110 gets the rate of its group, so the rates can be multiplied with the single-age population later. Ages under 45 get no rate (the study only covered 45+, and low-energy hip fractures are very rare at younger ages).

**Gates:** the evidence test agrees with the 80+ reading · rates rise with age for both sexes · every age 45–110 has exactly one rate per sex · confidence limits surround each rate · saved files read back identical.

In [10]:
# Starting: I take the age-group rows of the paper's table (without the TOTAL row)
r = rates[rates['age_group'] != 'TOTAL'].copy()

# Now I test what the last row really is, using the census age structure as evidence
paper_ratio = (r.iloc[-1]['population_2017_male'] + r.iloc[-1]['population_2017_female']) / \
              (r.iloc[-2]['population_2017_male'] + r.iloc[-2]['population_2017_female'])
saudi = pop[pop['nationality'] == 'saudi']
p75_79 = saudi[saudi['age'].between(75, 79)]['population'].sum()
ratio_80_84 = saudi[saudi['age'].between(80, 84)]['population'].sum() / p75_79
ratio_80plus = saudi[saudi['age'] >= 80]['population'].sum() / p75_79
print(f'Paper: last row / 75-79 row = {paper_ratio:.2f}')
print(f'Census: 80-84 / 75-79 = {ratio_80_84:.2f}   |   80+ / 75-79 = {ratio_80plus:.2f}')
gate(abs(paper_ratio - ratio_80plus) < abs(paper_ratio - ratio_80_84),
     'evidence test: the paper\'s last row behaves like an open 80+ group, not 80-84')

# Now I relabel the last row as 80+ (open-ended up to the oldest age in the census, 110)
r.loc[r.index[-1], ['age_group', 'age_end']] = ['Y080+', 110]
r['age_start'] = r['age_start'].astype(int)
r['age_end'] = r['age_end'].astype(int)

# Now I reshape into one row per age group and sex, with the rate and its 95% confidence limits
rows = []
for sex in ['male', 'female']:
    part = r[['age_group', 'age_start', 'age_end']].copy()
    part['sex'] = sex
    part['rate_per_100k']  = r[f'incidence_per_100000_{sex}']
    part['ci95_low']       = r[f'ci95_low_{sex}']
    part['ci95_high']      = r[f'ci95_high_{sex}']
    part['fractures_2yrs'] = r[f'fractures_2017_2018_{sex}']
    rows.append(part)
rates_clean = pd.concat(rows, ignore_index=True)

# Now I check the rates make clinical sense: they rise with age, and each sits inside its confidence interval
for sex in ['male', 'female']:
    s = rates_clean[rates_clean['sex'] == sex]['rate_per_100k']
    gate(s.is_monotonic_increasing, f'{sex} hip fracture rates rise with every older age group')
gate(((rates_clean['ci95_low'] <= rates_clean['rate_per_100k']) &
      (rates_clean['rate_per_100k'] <= rates_clean['ci95_high'])).all(), 'every rate lies inside its 95% confidence interval')
gate(rates_clean['fractures_2yrs'].sum() == 684, 'the clean table still holds all 684 fractures')

# Now I give every single age from 45 to 110 the rate of its group, so it can be multiplied with the population
single = []
for _, row in rates_clean.iterrows():
    for age in range(row['age_start'], row['age_end'] + 1):
        single.append({'sex': row['sex'], 'age': age, 'age_group': row['age_group'],
                       'rate_per_100k': row['rate_per_100k'],
                       'ci95_low': row['ci95_low'], 'ci95_high': row['ci95_high']})
rates_single = pd.DataFrame(single)
gate(len(rates_single) == 66 * 2, f'single-age table has {len(rates_single)} rows = 66 ages (45-110) x 2 sexes')
gate(not rates_single.duplicated(['sex', 'age']).any(), 'every age 45-110 has exactly one rate per sex')

# Now I save both tables and read them back to prove the files on Drive are complete
for name, table in [('hipfracture_rates_by_group.csv', rates_clean), ('hipfracture_rates_by_single_age.csv', rates_single)]:
    path = f'{CLEAN}/{name}'
    table.to_csv(path, index=False)
    gate(pd.read_csv(path).equals(table), f'{name} saved and read back identical ({len(table)} rows)')

# Finally I show the clean rates side by side
show = rates_clean.pivot_table(index='age_group', columns='sex', values='rate_per_100k')[['male', 'female']]
print('\nAnnual hip fracture rate per 100,000 Saudi citizens (2017-18):')
print(show.to_string())
print(f'\nStep 3 complete: {len(GATE_LOG)} gates passed so far, 0 failed')

Paper: last row / 75-79 row = 1.22
Census: 80-84 / 75-79 = 0.66   |   80+ / 75-79 = 1.24
GATE PASSED: evidence test: the paper's last row behaves like an open 80+ group, not 80-84
GATE PASSED: male hip fracture rates rise with every older age group
GATE PASSED: female hip fracture rates rise with every older age group
GATE PASSED: every rate lies inside its 95% confidence interval
GATE PASSED: the clean table still holds all 684 fractures
GATE PASSED: single-age table has 132 rows = 66 ages (45-110) x 2 sexes
GATE PASSED: every age 45-110 has exactly one rate per sex
GATE PASSED: hipfracture_rates_by_group.csv saved and read back identical (16 rows)
GATE PASSED: hipfracture_rates_by_single_age.csv saved and read back identical (132 rows)

Annual hip fracture rate per 100,000 Saudi citizens (2017-18):
sex         male  female
age_group               
Y045-049     2.8     2.2
Y050-054    18.0    17.0
Y055-059    21.0    39.0
Y060-064    37.0    55.0
Y065-069    79.0   115.0
Y070-074   16

## Step 4 – Clean the remaining tables, then the notebook gate
Three small tables are tidied for the model notebook:
- **UN population 2015–2030** (Saudi Arabia, all residents): one row per year × sex × 5-year age group. Used in Notebook 2 to compare with the paper's published estimates, which were also based on UN population figures.
- **Fracture sites** (Saudi patients in the study): neck of femur 43%, pertrochanteric 47%, subtrochanteric 10%.
- **Published projections** (paper, Table 2): 2015–2050, men and women.

### Notebook gate
Finally, every clean file is **read back from Google Drive** and checked against the **raw files read fresh from Drive** – nothing from memory is reused. If any clean file disagrees with its raw source, the notebook stops. This is the proof that Notebook 1 changed the *shape* of the data, never the numbers.

In [11]:
# Starting: I tidy the UN population into year x sex x 5-year age group
un_clean = (un.rename(columns={'Time': 'year', 'Sex': 'sex', 'AgeStart': 'age_start', 'Value': 'population'})
              [['year', 'sex', 'age_start', 'population']]
              .assign(sex=lambda d: d['sex'].str.lower())
              .sort_values(['year', 'sex', 'age_start'])
              .reset_index(drop=True))
gate(un_clean['population'].sum() == un['Value'].sum(), 'UN: total population unchanged by tidying')

# Now I tidy the fracture sites into shares that add to 1
sites_clean = sites.assign(share=sites['percent'] / 100)
gate(abs(sites_clean['share'].sum() - 1) < 1e-9, 'fracture-site shares add up to 1')

# Now I turn the published projections into one row per year and sex
proj_clean = (table2.reset_index()
                .melt(id_vars='sex', var_name='year', value_name='hip_fractures')
                .assign(year=lambda d: d['year'].str[1:].astype(int)))
gate(len(proj_clean) == 3 * 5, 'published projections: 3 rows (men, women, total) x 5 years')

# Now I save the three tables
saves = {'un_population_2015_2030.csv': un_clean,
         'hipfracture_sites.csv': sites_clean,
         'published_projections.csv': proj_clean}
for name, table in saves.items():
    table.to_csv(f'{CLEAN}/{name}', index=False)

# ===================== NOTEBOOK GATE =====================
# Now I re-read EVERY raw and clean file fresh from Drive, so nothing from memory is reused
print('\n================ NOTEBOOK 1 GATE ================')
raw_census = pd.read_csv(FILES['census'], encoding='utf-8-sig')
raw_census = raw_census[raw_census['Governorate Code'] == 'TOT_KSA']
raw_un     = pd.read_csv(FILES['un'])
raw_rates  = pd.read_csv(FILES['rates'])
raw_rates  = raw_rates[raw_rates['age_group'] != 'TOTAL'].reset_index(drop=True)
raw_sites  = pd.read_csv(FILES['sites'])
raw_t2     = pd.read_csv(FILES['table2'])

c_pop    = pd.read_csv(f'{CLEAN}/pop2022_by_age_sex_nationality.csv')
c_group  = pd.read_csv(f'{CLEAN}/hipfracture_rates_by_group.csv')
c_single = pd.read_csv(f'{CLEAN}/hipfracture_rates_by_single_age.csv')
c_un     = pd.read_csv(f'{CLEAN}/un_population_2015_2030.csv')
c_sites  = pd.read_csv(f'{CLEAN}/hipfracture_sites.csv')
c_proj   = pd.read_csv(f'{CLEAN}/published_projections.csv')

# 1. Census: every age, sex and nationality cell in the clean file equals the raw census
raw_cells = (raw_census.assign(age=raw_census['Age'].str[1:].astype(int),
                               sex=raw_census['Gender'].str.lower(),
                               nationality=raw_census['Citizenship'].str.lower())
                       .set_index(['nationality', 'sex', 'age'])['Observation value'].sort_index())
clean_cells = c_pop.set_index(['nationality', 'sex', 'age'])['population'].sort_index()
gate((raw_cells.values == clean_cells.values).all() and len(raw_cells) == len(clean_cells),
     f'NB GATE census: all {len(clean_cells)} clean cells equal the raw census, cell by cell')

# 2. Rates: every clean rate and fracture count equals the raw paper table
for sex in ['male', 'female']:
    g = c_group[c_group['sex'] == sex].reset_index(drop=True)
    gate((g['rate_per_100k'] == raw_rates[f'incidence_per_100000_{sex}']).all() and
         (g['fractures_2yrs'] == raw_rates[f'fractures_2017_2018_{sex}']).all(),
         f'NB GATE rates: {sex} rates and fracture counts equal the paper, row by row')

# 3. Single-age rates: every single age carries exactly its group's rate
merged = c_single.merge(c_group[['sex', 'age_group', 'rate_per_100k']], on=['sex', 'age_group'], suffixes=('', '_grp'))
gate(len(merged) == len(c_single) and (merged['rate_per_100k'] == merged['rate_per_100k_grp']).all(),
     'NB GATE rates: every single age 45-110 carries its group\'s rate')

# 4. UN: totals for every year and sex equal the raw UN file
raw_un_tot = raw_un.groupby(['Time', 'Sex'])['Value'].sum().values
cln_un_tot = c_un.groupby(['year', 'sex'])['population'].sum().values
gate(len(raw_un_tot) == 32 and (abs(raw_un_tot - cln_un_tot) < 1e-6).all(),
     'NB GATE UN: all 32 year x sex totals equal the raw UN file')

# 5. Sites and published projections equal the paper
gate((c_sites['percent'] == raw_sites['percent']).all(), 'NB GATE sites: 43 / 47 / 10 percent equal the paper')
gate(c_proj['hip_fractures'].sum() == raw_t2[[c for c in raw_t2.columns if c.startswith('y')]].values.sum(),
     'NB GATE projections: every published number carried over')

# Finally I list what this notebook produced and the total number of checks
print('\nClean files in data/clean:')
for f in sorted(os.listdir(CLEAN)):
    print(f'   {f:45s} {os.path.getsize(f"{CLEAN}/{f}"):>8,} bytes')
print(f'\nNOTEBOOK 1 COMPLETE: {len(GATE_LOG)} gates passed, 0 failed. Raw and clean data agree.')

GATE PASSED: UN: total population unchanged by tidying
GATE PASSED: fracture-site shares add up to 1
GATE PASSED: published projections: 3 rows (men, women, total) x 5 years

================ NOTEBOOK 1 GATE ================
GATE PASSED: NB GATE census: all 444 clean cells equal the raw census, cell by cell
GATE PASSED: NB GATE rates: male rates and fracture counts equal the paper, row by row
GATE PASSED: NB GATE rates: female rates and fracture counts equal the paper, row by row
GATE PASSED: NB GATE rates: every single age 45-110 carries its group's rate
GATE PASSED: NB GATE UN: all 32 year x sex totals equal the raw UN file
GATE PASSED: NB GATE sites: 43 / 47 / 10 percent equal the paper
GATE PASSED: NB GATE projections: every published number carried over

Clean files in data/clean:
   backtest_2017_to_2022.csv                          803 bytes
   calibration_factor.csv                              66 bytes
   comparison_2015_with_paper.csv                     190 bytes
   hip_frac